# 04 · Cross-patient validation (the important notebook)

## Like you're five
Imagine a spelling test. If you saw the exact test questions while studying, a good score proves nothing. So we **teach the model with 49 patients and quiz it on the 50th** it has never met. Then we swap, 50 times.

Questions we answer:
- Does the BCI work on a **brand-new stroke patient**?
- Are some patients much harder than others?
- Which classifier is best?
- Can we trust the "confidence" number?

**Balanced accuracy** = average of "how often LEFT was right" and "how often RIGHT was right," so guessing one side can't hide.
**Chance line:** with only 40 trials, you need ~65% (26/40) before it is convincingly better than a coin flip.

In [ ]:
# --- Setup (works on your laptop AND on Google Colab) ---
import os, sys
if "google.colab" in sys.modules and not os.path.exists("src"):
    # Colab: get the project first, e.g.  !git clone <your-repo-url> && %cd stroke-rehab-bci/notebooks
    pass
sys.path.append("..")          # so we can "import src"
# !pip install -q moabb mne scikit-learn pandas matplotlib   # uncomment on Colab
import numpy as np, pandas as pd, matplotlib.pyplot as plt


In [ ]:
from src.preprocessing import load_liu2024, encode_labels
from src.models import get_models
from src.evaluation import *
N_SUBJECTS = 10   # <- set to None for the full 50-patient run (slow!)
subs = None if N_SUBJECTS is None else list(range(1, N_SUBJECTS + 1))
X, y_text, meta = load_liu2024(subjects=subs)
y, enc = encode_labels(y_text); groups = meta["subject"].to_numpy()
print(X.shape, "patients:", len(np.unique(groups)))

In [ ]:
results, all_trials = {}, {}
for name, model in get_models().items():
    s, t = leave_one_patient_out(model, X, y, groups)
    results[name] = add_chance_flag(s); all_trials[name] = t
    s.to_csv(f"../results/{name.replace(' ', '_').replace('+','plus')}_per_patient.csv", index=False)
    print(f"{name:18s} mean={s.balanced_accuracy.mean():.3f}  SD={s.balanced_accuracy.std():.3f}  "
          f"patients above chance: {results[name].above_chance.sum()}/{len(s)}")

### Figures

In [ ]:
best = "CSP + LDA"
plot_patient_scores(results[best], title=f"{best}: balanced accuracy per held-out patient",
                    save_path="../figures/patient_scores.png")
plot_confusion(all_trials[best], title=f"{best}", save_path="../figures/confusion_matrix.png")
plot_classifier_comparison(results, save_path="../figures/classifier_comparison.png")
results[best].sort_values("balanced_accuracy")

### Can we trust "confidence"?
If the model says "80% sure," is it right about 80% of the time? Compare `claimed` to `actually_correct`. If they differ a lot, the number on the rehab screen is not trustworthy and should be shown with a warning.

In [ ]:
calibration_table(all_trials["CSP + LogReg"])

### Extension: mu-only vs beta-only vs combined
(Brief section 19/25.) Only the frequency band changes.

In [ ]:
bands = {"mu 8-13": (8, 13), "beta 13-30": (13, 30), "combined 8-30": (8, 30)}
band_res = {}
for bname, (lo, hi) in bands.items():
    Xb, yb, mb = load_liu2024(subjects=subs, fmin=lo, fmax=hi)
    s, _ = leave_one_patient_out(get_models()["CSP + LDA"], Xb, enc.transform(yb), mb["subject"].to_numpy())
    band_res[bname] = s; print(bname, round(s.balanced_accuracy.mean(), 3))
plot_classifier_comparison(band_res, save_path="../figures/band_comparison.png")

> Record the numbers you actually get here. Do **not** copy numbers from anywhere else into your README or résumé.